# AF2FFAB2 paired confirmation — seed 2026
Menjalankan AF2FFA0 lalu AF2FFAB2 secara berurutan dari checkpoint AF2 seed-matched. Notebook seed 123 dan 2026 boleh berjalan paralel di akun berbeda. Test tidak tersedia.

In [ ]:
SEED=2026
assert SEED in (123,2026)
BRANCH='codex/af2-feature-frequency-adapter'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import importlib,json,os,shutil,subprocess,sys,tarfile,time,torch
from pathlib import Path
assert torch.cuda.is_available(),'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
for module_name in list(sys.modules):
    if module_name=='coffee_detector' or module_name.startswith('coffee_detector.'): sys.modules.pop(module_name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('GPU:',torch.cuda.get_device_name(0),'| SEED:',SEED)

In [ ]:
from coffee_detector.drive_project import resolve_drive_project_root,require_project_artifact
AF2_REL=f'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed{SEED}/weights/best.pt'
SCREEN_REL='experiments/faruq-v3-af2-ffa-gradient-matched-bound-v1/val_reports/af2_ffa_gradient_matched_seed42_decision.json'
C42_REL='experiments/faruq-v3-af2-feature-frequency-adapter-v1/val_reports/AF2FFA0_seed42_result.json'
B42_REL='experiments/faruq-v3-af2-ffa-gradient-matched-bound-v1/val_reports/AF2FFAB2_seed42_result.json'
REQ=('bundles/faruq-development-v3-grouped.tar',AF2_REL,SCREEN_REL,C42_REL,B42_REL)
PROJECT=resolve_drive_project_root(required_relative_paths=REQ)
ARCHIVE=require_project_artifact(PROJECT,REQ[0]); AF2=require_project_artifact(PROJECT,AF2_REL)
SCREEN=require_project_artifact(PROJECT,SCREEN_REL); C42=require_project_artifact(PROJECT,C42_REL); B42=require_project_artifact(PROJECT,B42_REL)
screen=json.loads(SCREEN.read_text()); assert screen['decision']=='RETAIN_PARETO' and screen['test_opened'] is False
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'faruq_grouped_summary.json').is_file():
    if DATA.exists(): shutil.rmtree(DATA)
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file() and not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-ffa-gradient-matched-paired-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
STATIC=OUTPUT/f'static_audit_seed{SEED}.json'
print('AF2:',AF2); print('OUTPUT:',OUTPUT)

In [ ]:
from coffee_detector.af2_ffa.audit import run_af2_ffa_static_audit
audit=run_af2_ffa_static_audit(AF2,STATIC,device='cuda:0')
print('GRADIENTS:',{k:audit['records'][k]['initial_amplitude_gradient_mean'] for k in ('AF2FFA1','AF2FFAB1','AF2FFAB2')})
print('STATIC DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit gagal.'

In [ ]:
for ARM in ('AF2FFA0','AF2FFAB2'):
    RESULT=OUTPUT/'val_reports'/f'{ARM}_seed{SEED}_result.json'; LOG=OUTPUT/f'{ARM}_seed{SEED}_run.log'
    if RESULT.is_file():
        print('REUSE COMPLETE:',RESULT); continue
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--screening-decision',str(SCREEN),'--output-root',str(OUTPUT),'--seed',str(SEED),'--device','0','--authorize-training']
    print('START/RESUME:',ARM,'seed',SEED,'| log=',LOG,flush=True)
    with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    seen=None
    while process.poll() is None:
        csv=OUTPUT/ARM/f'{ARM}_seed{SEED}'/'results.csv'; epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epochs!=seen: print(f'{ARM} seed {SEED}: {epochs}/30 epoch tercatat',flush=True); seen=epochs
        time.sleep(60)
    if process.returncode:
        print('\n'.join(LOG.read_text(errors='replace').splitlines()[-150:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
    assert RESULT.is_file(),RESULT
    payload=json.loads(RESULT.read_text()); print(ARM,{k:payload['metrics'][k] for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})

In [ ]:
C=(C42,OUTPUT/'val_reports/AF2FFA0_seed123_result.json',OUTPUT/'val_reports/AF2FFA0_seed2026_result.json')
B=(B42,OUTPUT/'val_reports/AF2FFAB2_seed123_result.json',OUTPUT/'val_reports/AF2FFAB2_seed2026_result.json')
if all(path.is_file() for path in C+B):
    from coffee_detector.experiments.run_faruq_v3_af2_ffa_paired_confirmation import run_faruq_v3_af2_ffa_paired_confirmation
    SUMMARY=OUTPUT/'val_reports/af2_ffa_b2_paired_confirmation.json'
    decision=run_faruq_v3_af2_ffa_paired_confirmation(C,B,SUMMARY)
    print('FINAL:',decision['decision']); print('NEXT:',decision['next']); print('SUMMARY:',SUMMARY)
else:
    print('MENUNGGU NOTEBOOK PASANGAN:',{str(path):path.is_file() for path in C+B})
print('Jangan membuka test.')